# Notebook 02: Change Detection Baseline

Evaluates the rule-based change detectors (generic, construction, deforestation, fire, flood) on a sample region and visualises results.

In [ ]:
import sys; sys.path.insert(0, '..')
from dotenv import load_dotenv; load_dotenv('../.env')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from apps.api.services.imagery import SentinelHubFetcher
from apps.api.services.change.generic import GenericChangeDetector
from apps.api.services.change.construction import ConstructionDetector
from apps.api.services.change.deforestation import DeforestationDetector
from apps.api.services.change.fire import FireDetector
from apps.api.services.classifier import EventClassifier

In [ ]:
# Bellary region — known for mining and construction activity
BBOX = [76.7, 14.8, 77.2, 15.3]
BEFORE_DATE = ('2023-01-01', '2023-01-31')
AFTER_DATE  = ('2024-01-01', '2024-01-31')

fetcher = SentinelHubFetcher()
before = fetcher.get_sentinel2_composite(BBOX, *BEFORE_DATE)
after  = fetcher.get_sentinel2_composite(BBOX, *AFTER_DATE)
print(f'Imagery loaded: {before.shape}')

In [ ]:
# Run all detectors
detectors = {
    'Generic':       GenericChangeDetector(),
    'Construction':  ConstructionDetector(),
    'Deforestation': DeforestationDetector(),
    'Fire':          FireDetector(),
}

results = {}
for name, det in detectors.items():
    mask, conf = det.detect(before, after)
    results[name] = (mask, conf)
    pct = 100 * mask.mean()
    print(f'{name:15s}: confidence={conf:.3f}  changed_pixels={pct:.2f}%')

In [ ]:
# Visualise masks
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

def to_rgb(arr):
    rgb = arr[:, :, [3, 2, 1]]
    return np.power(np.clip(rgb, 0, 0.3) / 0.3, 0.4)

axes[0].imshow(to_rgb(before)); axes[0].set_title('Before'); axes[0].axis('off')
axes[1].imshow(to_rgb(after));  axes[1].set_title('After');  axes[1].axis('off')

for i, (name, (mask, conf)) in enumerate(results.items(), start=2):
    axes[i].imshow(to_rgb(after))
    axes[i].imshow(mask, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
    axes[i].set_title(f'{name} (conf={conf:.2f})')
    axes[i].axis('off')

plt.suptitle('Change Detection Baseline — Bellary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('02_change_detection_baseline.png', dpi=150)
plt.show()

In [ ]:
# Run the full classifier pipeline
classifier = EventClassifier(
    detection_types=['construction', 'deforestation', 'fire', 'solar'],
    bbox=BBOX,
)
result = classifier.classify(before, after)

if result:
    print(f'Best match: {result.detected_type} (confidence={result.confidence:.3f})')
    print(f'Centroid: lat={result.lat}, lon={result.lon}')
    print(f'All scores: {result.all_scores}')
else:
    print('No significant change detected above threshold')